# Basic snippets for langgraph
(only for prototyping)

## Initialisierung eines LLMs, hier Gemma4B über Ollama.

In [1]:
from dataclasses import dataclass, field
from typing import Any

from langchain_ollama import ChatOllama


@dataclass(frozen=True)
class OllamaConfig:
    model: str = "gemma4:12b"
    base_url: str = "http://localhost:11434"
    temperature: float = 0.7
    max_tokens: int = 4096
    timeout: int = 120
    options: dict[str, Any] = field(default_factory=dict)


def make_llm(config: OllamaConfig) -> ChatOllama:
    """Return a ChatOllama instance for the given config."""
    return ChatOllama(
        model=config.model,
        base_url=config.base_url,
        temperature=config.temperature,
        num_predict=config.max_tokens,
        timeout=config.timeout,
        **config.options,
    )


def make_llm_with_tools(config: OllamaConfig, tools: list) -> ChatOllama:
    """Return a ChatOllama instance with tools bound."""
    return make_llm(config).bind_tools(tools)


In [8]:
config = OllamaConfig()
llm = make_llm(config)

question = """
    Welchen Wert hat die Fallbeschleunigung der Erde in m/s^2? 
    Bitte gib das Ergebnis als json im nachfolgenden Format aus:
    {Beschleunigung: <hier ergebnis in m/s^2 einfügen>"""
response = llm.invoke(question)

print(response.content)

```json
{
  "Beschleunigung": 9.81
}
```


## Beispiel mit Tool-Binding

In [15]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

@tool
def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"Sunny in {city}"

from langchain_core.messages import HumanMessage

model = make_llm(config)
model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(content="What's the weather in Berlin?")]

# Schritt 1: Modell antwortet mit Tool-Call (content ist hier leer — normal)
response = model_with_tools.invoke(messages)
print("Tool-Calls:", response.tool_calls)

# Schritt 2: Tool tatsaechlich ausfuehren und Ergebnis als ToolMessage zurueckspielen
for tc in response.tool_calls:
    if tc["name"] == "get_weather":
        result = get_weather.invoke(tc["args"])
        messages += [response, ToolMessage(content=result, tool_call_id=tc["id"])]

# Schritt 3: Modell formuliert jetzt den finalen Content
final = model_with_tools.invoke(messages)
print("Antwort:", final.content)

Tool-Calls: [{'name': 'get_weather', 'args': {'city': 'Berlin'}, 'id': '41aebea1-a679-457e-b209-78e1b4901529', 'type': 'tool_call'}]
Antwort: The weather in Berlin is currently sunny.


In [16]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'city': 'Berlin'},
  'id': '41aebea1-a679-457e-b209-78e1b4901529',
  'type': 'tool_call'}]

In [18]:
result = get_weather.invoke("{'city': 'Berlin'}")
result

"Sunny in {'city': 'Berlin'}"

### Umsetzung mit Langgraph

In [25]:
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"Sunny in {city}"


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


model = ChatOllama(model="gemma4:12b", base_url="http://localhost:11434").bind_tools([get_weather])


def agent(state: AgentState) -> dict:
    """Schritte 1 + 3: Modell mit dem aktuellen Nachrichtenverlauf fragen."""
    response = model.invoke(state["messages"])
    return {"messages": [response]}


workflow = StateGraph(AgentState)
workflow.add_node("agent", agent)                       # Schritte 1 + 3: Modell fragen
workflow.add_node("tools", ToolNode([get_weather]))    # Schritt 2: Tool ausfuehren
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)  # Tool-Calls? -> tools, sonst END
workflow.add_edge("tools", "agent")                    # Ergebnis zurueckspielen, Loop von vorn

app = workflow.compile()

result = app.invoke({"messages": [HumanMessage(content="What's the weather in Berlin?")]})
print(result["messages"][-1].content)

The weather in Berlin is currently sunny.


In [26]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent


@tool
def search_web(query: str) -> str:
    """Search the web for a query."""
    return f"Results for {query}: [Mocked Web Search]"


@tool
def calculate(x: int, y: int) -> int:
    """Calculate x + y."""
    return x + y


model = ChatOllama(model="gemma4:12b")
agent = create_agent(model, [search_web, calculate])

result = agent.invoke({"messages": [HumanMessage(content="What is 5 + 7?")]})
print(result["messages"][-1].content)

The answer is 12.
